# E2 -- 40-mode Gaussian mixture (2D): plot notebook

**This notebook only reads saved results.**

It must not call a sampler, a PT tuner, an LSC quadrature refinement, a `dt` refinement, or a reference builder, and it must not recompute any official metric. Every number drawn here already exists in a run's `metrics_timeseries.csv` or `cost_timeseries.csv`, written by `E2_mog40_run.ipynb` at run time.

Scatter, CDF, histogram, and KDE panels are **display only**. They visualise the saved sample snapshots and **never override, correct, or stand in for** the numbers in `metrics_timeseries.csv`. If a picture and a saved metric disagree, the saved metric is the result.

Method colours, markers, and display names come from `configs/registry.yaml`; which runs to draw and how to lay them out comes from `configs/plots/manuscript.yaml`. Neither table is redefined here.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "..")  # importable when launched from notebooks/

from src.catalog import select_runs
from src.plotting import (curve_figure, load_plot_config, load_runs,
                          save_figure, snapshot_figure)

REPO_ROOT = Path("..")
EXPERIMENT_ID = "E2"

plot_config = load_plot_config(REPO_ROOT / "configs" / "plots" / "manuscript.yaml")
defaults = plot_config["defaults"]
spec = plot_config[EXPERIMENT_ID]
figures = spec["figures"]

EXPERIMENT_DIR = REPO_ROOT / "results" / spec["experiment_key"]
OUTPUT_DIR = REPO_ROOT / defaults["output_root"] / spec["experiment_key"]
FORMATS = tuple(defaults["output_formats"])  # png, pdf, svg, tiff
FIGURES = {}

print(f"{EXPERIMENT_ID}: {len(figures)} specified figures -> {OUTPUT_DIR}")

## What is plottable

Load the derived catalog (rebuilding it from the manifests if it is missing) and list the runs it admits, so it is visible up front which methods, variants, and step sizes this notebook can actually draw. Nothing is run here; this is a directory listing.

In [ ]:
runs = select_runs(EXPERIMENT_DIR, latest_only=defaults["latest_run_only"])

print(f"{len(runs)} plottable runs\n")
print(f"{'method':<12}{'variant label':<34}{'tame':<7}{'dt':<10}run id")
for row in runs:
    print(f"{row['method']:<12}{row['variant_label']:<34}"
          f"{str(row['tame']):<7}{str(row['dt']):<10}{row['run_id']}")

### Figure E2.1 -- target contours with method samples

A 2x3 small-multiple: the exact target contour with one method's samples per panel (ULA, ULD, FLA, PT, LSC-CP, LSC-CP-RA), all at **one matched simulation time**. Every panel shares the same axes, the same contour levels, and the same point count drawn by the same deterministic subsampling rule, so the panels are visually comparable.

In [ ]:
figure_spec = figures["E2.1_scatter_matched_time"]

FIGURES["E2.1_scatter_matched_time"] = snapshot_figure(
    load_runs(EXPERIMENT_DIR, figure_spec), figure_spec)
FIGURES["E2.1_scatter_matched_time"]

### Figure E2.2 -- mode coverage and mode weights

Exactly **three** panels:

1. **EMC** against simulation time and against FEE, with the frozen **EMC\* reference line** -- the hard-assignment descriptor value estimated once from the large exact bank. The reference line is EMC\*, **not 1**.
2. **Mode-weight Jensen-Shannon divergence** against simulation time and against FEE.
3. **Per-mode occupancy ratio** $\hat p_k / p^\star_k$ at the chosen matched times, with a reference line at **1**.

All three read saved columns; none of them is recomputed here.

In [ ]:
figure_spec = figures["E2.2_mode_metrics"]

FIGURES["E2.2_mode_metrics"] = curve_figure(
    load_runs(EXPERIMENT_DIR, figure_spec), figure_spec)
FIGURES["E2.2_mode_metrics"]

### Figure E2.3 -- supplement

Supplementary two-sample metrics for E2 (SW$_2$, biased and unbiased MMD$^2$) across the full method set. Supplement only: the main-text mode claims rest on Figure E2.2.

In [ ]:
figure_spec = figures["E2.3_supplement"]

FIGURES["E2.3_supplement"] = curve_figure(
    load_runs(EXPERIMENT_DIR, figure_spec), figure_spec)
FIGURES["E2.3_supplement"]

### Figure E2.4 -- LSC score potential-evaluation cost

The LSC-only cost figure: full LSC-CP against LSC-CP-RA(A). The x axis counts LSC score potential evaluations only and is not a complete computational cost.

In [ ]:
figure_spec = figures["E2.4_lsc_score_cost"]

FIGURES["E2.4_lsc_score_cost"] = curve_figure(
    load_runs(EXPERIMENT_DIR, figure_spec), figure_spec)
FIGURES["E2.4_lsc_score_cost"]

## Canonical, tamed, and paired views

The main curve figure is regenerated three times, from the same saved runs: **canonical only**, **tamed only**, and the **paired** canonical-versus-tamed overlay.

The convention, taken from `configs/registry.yaml` and the plot defaults:

* the **method** sets the **colour**, and taming never changes it;
* **canonical** is a **solid** line;
* **tamed** is a **dashed** line;
* **hyperparameter values** are distinguished by **marker**, and the value is written into the legend label.

So colour answers "which method", line style answers "tamed or not", and marker answers "which hyperparameter value".

In [ ]:
MAIN_CURVE_FIGURE = "E2.2_mode_metrics"

for view in defaults["tame_views"]:
    view_spec = {**figures[MAIN_CURVE_FIGURE], "tame_view": view}
    FIGURES[f"{MAIN_CURVE_FIGURE}__{view}"] = curve_figure(
        load_runs(EXPERIMENT_DIR, view_spec), view_spec)

print("tame views:", list(defaults["tame_views"]))

## Export

Every figure built above is written to **PNG, PDF, SVG, and TIFF** under `figures/<experiment key>/`, from the format list in the plot defaults. Re-running this cell overwrites the files in place; it never touches anything under `results/`.

In [ ]:
for name, figure in FIGURES.items():
    save_figure(figure, name, OUTPUT_DIR, formats=FORMATS)
    print(f"saved {name}  [{', '.join(FORMATS)}]")

print(f"\n{len(FIGURES)} figures written under {OUTPUT_DIR}")